<div dir="rtl" style="text-align:right; font-family:'Segoe UI', 'Arial Hebrew', Arial, sans-serif;line-height:1.7;">

<h1 style="color:#1f3a68;">NetSec Dashboard — HDBSCAN + Hopkins</h1>

<h3 style="color:#555;">תצוגה מקדימה לשינויים במחברת הראשית</h3>

מסמך זה מציג שלוש הוספות לפרויקט: <b>חישוב Hopkins statistic</b>, מעבר ל-<b>HDBSCAN</b> כקלאסטרינג ראשי, ו-<b>fallback אוטומטי ל-DBSCAN</b> כשהמשני נכשל או לא זמין. בכל סעיף מוצגים שלושה רכיבים: הסבר תיאורטי, הקוד הקיים (BEFORE), והקוד המוצע (AFTER). שום קוד לא מורץ — זוהי השוואה בלבד.

</div>

<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

## תוכן עניינים

1. סקירת הפרויקט וההקשר
2. שלושת המודלים — IsolationForest, DBSCAN, LSTM
3. DBSCAN לעומק וההקשר ל-Silhouette
4. Hopkins Statistic — מה זה ולמה צריך אותו
5. למה לעבור ל-HDBSCAN ולמה fallback ל-DBSCAN
6. **שינוי בקוד #1** — Cell 4 (bootstrap)
7. **שינוי בקוד #2** — Cell 12 (`run_ml_on_session`)
8. **שינוי בקוד #3** — Cell 47 (Model Diagnostics UI)
9. ערובות תאימות לאחור
10. שש שאלות מוכנות למרצה

</div>


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

## 1. סקירת הפרויקט וההקשר

זהו Dashboard פורנזי לניתוח תעבורת רשת מבוסס **Wireshark + Dash + scikit-learn + PyTorch**.
הוא טוען קבצי `.pcapng` (או מקליט תעבורה חיה דרך `tshark`), מחלץ פיצ'רים פר-IP,
ומריץ במקביל שלושה מודלי **למידה לא-מפוקחת** + שתי שכבות חוקים דטרמיניסטיות.

**מה הוא מבצע — בשורות תמציתיות:**

- חילוץ 7 פיצ'רים פר-IP: `mean_len, std_len, count, burst_score, unique_dsts, syn_count, rst_count`.
- נירמול עם `StandardScaler` (קריטי כי הפיצ'רים בסקאלות שונות).
- שלושה מודלי ML במקביל: **IsolationForest**, **DBSCAN**, **LSTM**.
- שכבות חוקים: זיהוי TCP SYN scan/flood ו-ARP spoofing / DNS tunneling.
- סיווג מכשירים ל-12 קטגוריות במנוע 3-שכבתי.
- השוואה צד-בצד של שני סשנים (S1 ↔ S2).

**הבעיה שאני פותר כעת:** במצב הקיים, DBSCAN לעיתים מחזיר אשכול יחיד + מעט noise,
מה שמוביל ל-`Silhouette = n/a` ולחוסר ודאות לגבי איכות הקלאסטרינג. ההוספה של Hopkins
ושל HDBSCAN נועדה לתת תשובה מבוססת לשאלה "האם בכלל יש מבנה אשכולי בנתונים, ואם כן —
האם DBSCAN באמת המודל הנכון?"

</div>


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

## 2. שלושת המודלים — IsolationForest, DBSCAN, LSTM

כל מודל תופס סוג שונה של אנומליה. השילוב ביניהם הוא ה-cross-validation הלא-מפוקח של הפרויקט.

<table dir="rtl" style="border-collapse:collapse; width:100%;">
<thead>
<tr style="background:#eef;">
<th style="border:1px solid #ccc; padding:6px;">מודל</th>
<th style="border:1px solid #ccc; padding:6px;">סוג אנומליה שתופס</th>
<th style="border:1px solid #ccc; padding:6px;">הפלט שלו</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #ccc; padding:6px;"><b>IsolationForest</b></td>
<td style="border:1px solid #ccc; padding:6px;">Outliers גלובליים פר-IP במרחב הפיצ'רים</td>
<td style="border:1px solid #ccc; padding:6px;">ציון רציף + תיוג בינארי</td>
</tr>
<tr>
<td style="border:1px solid #ccc; padding:6px;"><b>DBSCAN</b></td>
<td style="border:1px solid #ccc; padding:6px;">Outliers מקומיים מבוססי <b>צפיפות</b></td>
<td style="border:1px solid #ccc; padding:6px;">תווית אשכול (−1 = noise = חשוד)</td>
</tr>
<tr>
<td style="border:1px solid #ccc; padding:6px;"><b>LSTM</b></td>
<td style="border:1px solid #ccc; padding:6px;">אנומליות <b>זמניות / רצפיות</b></td>
<td style="border:1px solid #ccc; padding:6px;">שגיאת חיזוי (סף val_mean + 2σ)</td>
</tr>
</tbody>
</table>

### IsolationForest

בונה 200 עצי בידוד אקראיים. נקודה חריגה מבודדת אחרי מעט פיצולים; נקודה רגילה דורשת
הרבה פיצולים. ה-`contamination` (אחוז האנומליות הצפוי) **לא קבוע** — נבחר אוטומטית
ע"י sensitivity sweep על `[0.05, 0.10, 0.15]`.

### DBSCAN (זה שמשתנה — ראה סעיף 3 ו-5)

מקבץ נקודות לפי **צפיפות מקומית** במרחב הפיצ'רים. נקודות שאי-אפשר לקבץ אותן עם אף אחד
מקבלות תווית `−1` (noise) — וזהו אות אנומליה התנהגותי.

### LSTM (PyTorch)

רשת רקורנטית שמאומנת על רצפי גודל-פקטה ב-bins של שנייה (`SEQ_LEN=10`). מנבאת את הצעד
הבא; שגיאת חיזוי מעל הסף → אנומליה. early stopping עם `PATIENCE=2` מונע overfitting.

</div>


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

## 3. DBSCAN לעומק וההקשר ל-Silhouette

### איך DBSCAN עובד

DBSCAN שואל לכל נקודה: *"כמה שכנים יש לי ברדיוס `eps`?"*

- **Core point** — יש לפחות `min_samples` שכנים בתוך `eps` → שייכת לאשכול.
- **Border point** — בתוך `eps` של core אבל אין לה עצמה מספיק שכנים.
- **Noise point** — אין לה אף שכן ברדיוס → תווית `−1` → אנומליה.

### ההיפר-פרמטרים הקריטיים

- **`eps`** — נבחר אוטומטית ממרפק גרף ה-k-distance (נגזרת שנייה מינימלית).
  fallback ל-`1.3` רק כשיש פחות מ-4 IP-ים.
- **`min_samples=2`** — קבוע ונמוך בכוונה. ב-7 ממדים ו-50–150 IP, ערך גבוה יותר היה
  הופך כמעט הכל ל-noise.

### Silhouette Score והקשר ל-DBSCAN

מדד פנימי להערכת איכות אשכולות (לא דורש תוויות):

- `a(i)` — מרחק ממוצע של נקודה לשאר נקודות באשכול שלה (קוהזיה).
- `b(i)` — מרחק ממוצע לאשכול הקרוב השני (הפרדה).
- `s(i) = (b − a) / max(a, b)` בטווח `[−1, 1]`. גבוה = טוב.

**הבעיה שלי:** ב-DBSCAN, כשמתקבל אשכול יחיד + מעט noise — Silhouette **לא מוגדר**
(אין "אשכול אחר" לחשב אליו את b). זה מתבטא ב-`Silhouette = n/a` ב-Model Diagnostics,
וזו אחת הסיבות לחפש מדד שלם יותר (Hopkins) ואלגוריתם שמטפל בצפיפויות משתנות (HDBSCAN).

</div>


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

## 4. Hopkins Statistic — מה זה ולמה צריך אותו

Hopkins הוא מבחן סטטיסטי שבודק **האם הדאטה שלי מכיל נטייה לאשכולות בכלל**, או שהוא
פשוט רעש אחיד אקראי. זה צעד מקדים לכל ניסיון אשכול — בלעדיו, גם DBSCAN וגם HDBSCAN
יכולים להחזיר תוויות שרירותיות.

### אלגוריתם החישוב

1. בוחרים `m` נקודות אקראיות מתוך הדאטה (`X_i`).
2. מייצרים `m` נקודות **סינתטיות** אקראיות בתוך אותו טווח (`Y_i`).
3. מודדים מרחק לשכן הקרוב ביותר: `u_i` עבור הסינתטיות, `w_i` עבור האמיתיות.
4. **`H = Σu_i / (Σu_i + Σw_i)`**

### פרשנות הערכים

<table dir="rtl" style="border-collapse:collapse; width:100%;">
<thead>
<tr style="background:#eef;">
<th style="border:1px solid #ccc; padding:6px;">ערך H</th>
<th style="border:1px solid #ccc; padding:6px;">משמעות</th>
<th style="border:1px solid #ccc; padding:6px;">המלצה</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #ccc; padding:6px;">≈ 0.5</td>
<td style="border:1px solid #ccc; padding:6px;">אין מבנה — דאטה אחיד</td>
<td style="border:1px solid #ccc; padding:6px;">לסמוך על IsolationForest, להתעלם מתוויות הקלאסטרינג</td>
</tr>
<tr>
<td style="border:1px solid #ccc; padding:6px;">> 0.7</td>
<td style="border:1px solid #ccc; padding:6px;">נטייה ברורה לאשכולות</td>
<td style="border:1px solid #ccc; padding:6px;">שווה לאשכל; אם DBSCAN נכשל — HDBSCAN</td>
</tr>
<tr>
<td style="border:1px solid #ccc; padding:6px;">> 0.9</td>
<td style="border:1px solid #ccc; padding:6px;">מבנה חזק במיוחד</td>
<td style="border:1px solid #ccc; padding:6px;">HDBSCAN צפוי לחלץ אשכולות בבירור</td>
</tr>
</tbody>
</table>

המימוש המוצע — פונקציה נוספת מעל `run_ml_on_session`. הקוד המלא בסעיף 7 (Cell 12 AFTER),
אך הנה גרסה תמציתית לעיון מהיר:

</div>


In [ ]:
def hopkins_statistic(X, m=None, random_state=42):
    """H ~ 0.5 = random; H > 0.7 = cluster tendency."""
    import numpy as np
    from sklearn.neighbors import NearestNeighbors
    rng = np.random.default_rng(random_state)
    n, d = X.shape
    if m is None:
        m = max(5, int(0.1 * n))

    idx     = rng.choice(n, size=m, replace=False)
    sample  = X[idx]
    mins, maxs = X.min(axis=0), X.max(axis=0)
    synth   = rng.uniform(mins, maxs, size=(m, d))

    nbrs = NearestNeighbors(n_neighbors=2).fit(X)
    w, _ = nbrs.kneighbors(sample, n_neighbors=2)
    w    = w[:, 1]
    u, _ = nbrs.kneighbors(synth, n_neighbors=1)
    u    = u[:, 0]

    return float(u.sum() / (u.sum() + w.sum()))


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

## 5. למה לעבור ל-HDBSCAN ולמה fallback ל-DBSCAN

### למה HDBSCAN ראשי

DBSCAN משתמש ב-`eps` יחיד לכל המרחב — מתאים רק כשהצפיפות אחידה בכל הדאטה.
תעבורת רשת הטרוגנית במהותה: IoT עם דפוס אחד, browsers עם דפוס שני, שרתים עם שלישי —
לכל אחד "צפיפות" טבעית שונה. HDBSCAN פותר זאת: הוא הופך את DBSCAN למודל היררכי
שלא דורש `eps`, ומשתמש ב-`min_cluster_size` במקום זה.

**יתרונות נוספים של HDBSCAN:**

- מטפל בצפיפויות משתנות → מחלץ אשכולות שונים בעוצמות שונות בלי כיוון ידני.
- מחזיר `probabilities_` — ציון ביטחון פר-נקודה, מה שמשפר את ה-Model Agreement Matrix.
- פותר את כל הדילמה של "`eps` שונה לכל סשן" — אין כיוון מפורש בכלל.

### למה fallback ל-DBSCAN

- `hdbscan` היא ספריית Python שדורשת קומפילציית C (תלות ב-Cython). במחשבים מסוימים
  ההתקנה נכשלת.
- אם HDBSCAN רץ אבל מחזיר רק noise (כל הנקודות `−1`) — אין מה להציג, ועדיף לחזור
  ל-DBSCAN שלפחות יוצר אשכול גס אחד.
- שמירה על תאימות עם הדשבורד הקיים: כל הויזואליזציות צופות `cluster ∈ {−1, 0, 1, …}`
  — שני האלגוריתמים מקיימים זאת.

### כללי הבחירה (במימוש המוצע)

```
אם hdbscan זמין:
    הרץ HDBSCAN(min_cluster_size=max(3, 3% מה-IPs), min_samples=2)
    אם חזר עם ≥ 1 cluster תקין:
        אמץ → _cluster_algo = "HDBSCAN"
    אחרת:
        fallback ל-DBSCAN → _cluster_algo = "DBSCAN (fallback from HDBSCAN)"
אחרת:
    DBSCAN → _cluster_algo = "DBSCAN (hdbscan not installed)"
```

### איך זה ישתקף ב-Model Diagnostics

**לפני (היום):**

```
DBSCAN: eps=0.78 (k-distance elbow) - min_samples=2 - clusters=1 - noise=6 - silhouette=n/a
```

**אחרי, כש-HDBSCAN מצליח:**

```
Hopkins:    H = 0.78  ->  strong cluster tendency
Clustering (HDBSCAN): min_cluster_size=4 - min_samples=2 - clusters=3 - noise=4 - silhouette=0.42
```

**אחרי, כש-fallback ל-DBSCAN:**

```
Hopkins:    H = 0.48  ->  no structure (random)
Clustering (DBSCAN (fallback from HDBSCAN)): eps=0.78 - min_samples=2 - clusters=1 - noise=6 - silhouette=n/a
```

</div>


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

## 6. שינוי בקוד #1 — Cell 4 (bootstrap)

**מטרה:** להוסיף ניסיון התקנה של `hdbscan` כתלות אופציונלית, ולהגדיר דגל
`HDBSCAN_AVAILABLE` שישמש בהמשך לקבלת החלטה.

**מה לא משתנה:** רשימת ה-PKGS הקיימת, סדר ה-imports, טבלת הגרסאות, חיפוש tshark.

**מה מתווסף:** בלוק `try/except` קצר אחרי הלולאה הקיימת + import נוסף של
`sklearn.neighbors.NearestNeighbors` ברמת המודול (כדי שפונקציית Hopkins תייבא נקי).

### BEFORE — קטע ה-bootstrap הנוכחי

</div>


In [ ]:
import subprocess, sys, os

PKGS = {
    'numpy':                    'numpy',
    'pandas':                   'pandas',
    'torch':                    'torch',
    'scikit-learn':             'sklearn',
    'scapy':                    'scapy',
    'plotly':                   'plotly',
    'dash':                     'dash',
    'dash-bootstrap-components':'dash_bootstrap_components',
    'manuf':                    'manuf',
}
for pkg, imp in PKGS.items():
    try:
        __import__(imp)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import re, warnings, datetime, collections, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import sklearn
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
import scapy
from scapy.all import rdpcap, conf as scapy_conf
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output, State, ctx
import dash_bootstrap_components as dbc
import manuf

scapy_conf.verb = 0
warnings.filterwarnings("ignore")

# ... (rest of cell 4 unchanged — version table + tshark probe) ...


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

### AFTER — הקטע המוצע

ההבדל ממוקד: בלוק 8 שורות אחרי הלולאה הראשית של PKGS. אם ההתקנה של hdbscan נכשלת
מסיבה כלשהי (חוסר קומפיילר, הגבלות סביבה) — הסקריפט ממשיך, רק `HDBSCAN_AVAILABLE`
נשאר `False` והקלאסטרינג בהמשך משתמש ב-DBSCAN כמו היום.

</div>


In [ ]:
import subprocess, sys, os

PKGS = {
    'numpy':                    'numpy',
    'pandas':                   'pandas',
    'torch':                    'torch',
    'scikit-learn':             'sklearn',
    'scapy':                    'scapy',
    'plotly':                   'plotly',
    'dash':                     'dash',
    'dash-bootstrap-components':'dash_bootstrap_components',
    'manuf':                    'manuf',
}
for pkg, imp in PKGS.items():
    try:
        __import__(imp)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Optional: HDBSCAN for density-based clustering with variable densities.
# Install is best-effort — if it fails (e.g. no C compiler), the notebook
# still runs and the clustering step falls back to DBSCAN.
HDBSCAN_AVAILABLE = False
try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except ImportError:
    print('Installing hdbscan (optional, used as primary clusterer)...')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'hdbscan', '-q'])
        import hdbscan
        HDBSCAN_AVAILABLE = True
    except Exception as e:
        print(f'  hdbscan unavailable: {e}. Falling back to DBSCAN only.')

import re, warnings, datetime, collections, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import sklearn
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import scapy
from scapy.all import rdpcap, conf as scapy_conf
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output, State, ctx
import dash_bootstrap_components as dbc
import manuf

scapy_conf.verb = 0
warnings.filterwarnings("ignore")

# ... (rest of cell 4 unchanged — version table + tshark probe) ...


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

## 7. שינוי בקוד #2 — Cell 12 (`run_ml_on_session`)

זהו השינוי המהותי — התא המרכזי שמריץ את כל ה-ML פר-סשן.

**שינויים:**

1. **פונקציה חדשה `hopkins_statistic(X, m, random_state)`** למעלה, יחד עם `_hopkins_label(H)`.
2. **חישוב Hopkins מודפס** לכל סשן מיד אחרי `StandardScaler`.
3. **בלוק ה-DBSCAN הישן מוחלף** בלוגיקת בחירה דו-שלבית: HDBSCAN ראשון, DBSCAN fallback.
4. **שדות חדשים** נשמרים על dict הסשן: `_hopkins`, `_hopkins_label`, `_cluster_algo`,
   `_min_cluster_size`. השדות הקיימים (`_chosen_contamination`, `_eps_auto`, `_silhouette`,
   `_n_clusters`, `_n_noise`, `_min_samples`) נשארים.
5. **`ip_agg["cluster"]`** שומר על אותה סמנטיקה (`−1` = noise, אחר = cluster id) — כל
   הקוד התלוי בזה (Model Agreement Matrix, ויזואליזציות, scoring) ממשיך לעבוד בלי שינוי.
6. `probabilities_` של HDBSCAN נשמרים ב-`ip_agg["cluster_prob"]` כשהאלגוריתם זמין —
   זמין לויזואליזציות עתידיות, אופציונלי.

### BEFORE — הקוד הנוכחי של `run_ml_on_session`

</div>


In [ ]:
def run_ml_on_session(S):
    """Run IsolationForest + DBSCAN on a session's per-IP feature matrix. Mutates S['ip_agg'] in-place to add columns: iso_score, iso_flag, anomaly, cluster."""
    if S is None or S.get("ip_agg") is None or len(S["ip_agg"]) == 0:
        print("  (no ip_agg available - skipping ML)")
        return

    import numpy as np
    from sklearn.neighbors import NearestNeighbors
    from sklearn.ensemble import IsolationForest
    from sklearn.cluster import DBSCAN
    from sklearn.preprocessing import StandardScaler

    ip_agg = S["ip_agg"]
    FEATURE_COLS = ["mean_len","std_len","count","burst_score",
                    "unique_dsts","syn_count","rst_count"]
    X_raw = ip_agg[FEATURE_COLS].fillna(0).values
    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)

    print(f"[{S['label']}] Feature matrix: {X.shape[0]} IPs x {X.shape[1]} features")

    print(f"[{S['label']}] IsolationForest - contamination sensitivity analysis:")
    best_cont, best_score = 0.10, np.inf
    for cont in [0.05, 0.10, 0.15]:
        iso_tmp = IsolationForest(n_estimators=200, contamination=cont, random_state=42)
        iso_tmp.fit(X)
        scores_tmp = iso_tmp.decision_function(X)
        flagged    = scores_tmp[iso_tmp.predict(X) == -1]
        mean_score = flagged.mean() if len(flagged) else 0
        n_flagged  = (iso_tmp.predict(X) == -1).sum()
        print(f"  contamination={cont:.2f} -> {n_flagged:3d} IPs flagged | "
              f"mean anomaly score of flagged: {mean_score:.4f}")
        if mean_score < best_score:
            best_score, best_cont = mean_score, cont
    print(f"  => Selected contamination={best_cont:.2f}")

    iso = IsolationForest(n_estimators=200, contamination=best_cont, random_state=42)
    iso.fit(X)
    ip_agg["iso_score"] = iso.decision_function(X)
    ip_agg["iso_flag"]  = iso.predict(X)
    ip_agg["anomaly"]   = ip_agg["iso_flag"] == -1

    k = 2
    nbrs = NearestNeighbors(n_neighbors=k).fit(X)
    distances, _ = nbrs.kneighbors(X)
    k_dist = np.sort(distances[:, k-1])[::-1]
    if len(k_dist) >= 4:
        d1 = np.diff(k_dist)
        d2 = np.diff(d1)
        elbow_idx = int(np.argmin(d2)) + 1
        eps_auto  = float(round(k_dist[elbow_idx], 2))
    else:
        eps_auto = 1.3

    print(f"[{S['label']}] DBSCAN eps={eps_auto:.2f} (min_samples=2)")
    dbscan = DBSCAN(eps=eps_auto, min_samples=2)
    ip_agg["cluster"] = dbscan.fit_predict(X)

    # Cluster-quality diagnostics - stored so the dashboard can display them.
    from sklearn.metrics import silhouette_score
    _labels   = ip_agg["cluster"].values
    _nonnoise = _labels != -1
    _n_clusters = int(len(set(_labels[_nonnoise])))
    _n_noise    = int((_labels == -1).sum())
    try:
        if _nonnoise.sum() >= 2 and _n_clusters >= 2:
            _sil = float(silhouette_score(X[_nonnoise], _labels[_nonnoise]))
        else:
            _sil = None
    except Exception:
        _sil = None

    S["ip_agg"] = ip_agg
    S["_X"] = X
    S["_chosen_contamination"] = best_cont
    S["_eps_auto"]    = eps_auto
    S["_min_samples"] = 2
    S["_silhouette"]  = _sil
    S["_n_clusters"]  = _n_clusters
    S["_n_noise"]     = _n_noise
    print(f"[{S['label']}] DBSCAN clusters={_n_clusters} noise={_n_noise} "
          f"silhouette={('n/a' if _sil is None else round(_sil,3))}")
    print(f"[{S['label']}] Anomalies: {ip_agg['anomaly'].sum()} / {len(ip_agg)} | "
          f"Clusters: {ip_agg['cluster'].nunique()}")


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

### AFTER — הקוד המוצע

</div>


In [ ]:
def hopkins_statistic(X, m=None, random_state=42):
    """Hopkins statistic for cluster tendency.

    H ~ 0.5 : data is indistinguishable from uniform random — no structure
              to cluster; downstream clustering labels would be arbitrary.
    H > 0.7 : strong cluster tendency.
    H > 0.9 : very strong cluster tendency.

    Reference: Lawson & Jurs (1990).
    """
    import numpy as np
    from sklearn.neighbors import NearestNeighbors
    rng = np.random.default_rng(random_state)
    n, d = X.shape
    if n < 5:
        return None
    if m is None:
        m = max(5, int(0.1 * n))
    m = min(m, n - 1)

    idx    = rng.choice(n, size=m, replace=False)
    sample = X[idx]

    mins, maxs = X.min(axis=0), X.max(axis=0)
    synth = rng.uniform(mins, maxs, size=(m, d))

    nbrs = NearestNeighbors(n_neighbors=2).fit(X)
    w, _ = nbrs.kneighbors(sample, n_neighbors=2)
    w    = w[:, 1]                # skip the point itself
    u, _ = nbrs.kneighbors(synth, n_neighbors=1)
    u    = u[:, 0]

    denom = u.sum() + w.sum()
    if denom == 0:
        return None
    return float(u.sum() / denom)


def _hopkins_label(H):
    if H is None:                return "n/a (too few points)"
    if H >= 0.9:                 return "very strong cluster tendency"
    if H >= 0.7:                 return "strong cluster tendency"
    if H >= 0.55:                return "weak cluster tendency"
    return "no structure (random)"


def run_ml_on_session(S):
    """Run IsolationForest + (HDBSCAN with DBSCAN fallback) on a session's
    per-IP feature matrix. Mutates S['ip_agg'] in-place to add columns:
    iso_score, iso_flag, anomaly, cluster."""
    if S is None or S.get("ip_agg") is None or len(S["ip_agg"]) == 0:
        print("  (no ip_agg available - skipping ML)")
        return

    import numpy as np
    from sklearn.neighbors import NearestNeighbors
    from sklearn.ensemble import IsolationForest
    from sklearn.cluster import DBSCAN
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import silhouette_score

    ip_agg = S["ip_agg"]
    FEATURE_COLS = ["mean_len","std_len","count","burst_score",
                    "unique_dsts","syn_count","rst_count"]
    X_raw = ip_agg[FEATURE_COLS].fillna(0).values
    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)

    print(f"[{S['label']}] Feature matrix: {X.shape[0]} IPs x {X.shape[1]} features")

    # --- Hopkins statistic (cluster tendency) -------------------------------
    H = hopkins_statistic(X)
    H_label = _hopkins_label(H)
    print(f"[{S['label']}] Hopkins statistic = "
          f"{('n/a' if H is None else f'{H:.3f}')}  ->  {H_label}")

    # --- IsolationForest (unchanged) ----------------------------------------
    print(f"[{S['label']}] IsolationForest - contamination sensitivity analysis:")
    best_cont, best_score = 0.10, np.inf
    for cont in [0.05, 0.10, 0.15]:
        iso_tmp = IsolationForest(n_estimators=200, contamination=cont, random_state=42)
        iso_tmp.fit(X)
        scores_tmp = iso_tmp.decision_function(X)
        flagged    = scores_tmp[iso_tmp.predict(X) == -1]
        mean_score = flagged.mean() if len(flagged) else 0
        n_flagged  = (iso_tmp.predict(X) == -1).sum()
        print(f"  contamination={cont:.2f} -> {n_flagged:3d} IPs flagged | "
              f"mean anomaly score of flagged: {mean_score:.4f}")
        if mean_score < best_score:
            best_score, best_cont = mean_score, cont
    print(f"  => Selected contamination={best_cont:.2f}")

    iso = IsolationForest(n_estimators=200, contamination=best_cont, random_state=42)
    iso.fit(X)
    ip_agg["iso_score"] = iso.decision_function(X)
    ip_agg["iso_flag"]  = iso.predict(X)
    ip_agg["anomaly"]   = ip_agg["iso_flag"] == -1

    # --- Clustering: HDBSCAN primary, DBSCAN fallback -----------------------
    cluster_algo = None
    cluster_labels = None
    cluster_probs  = None
    eps_auto       = None
    min_cluster_sz = None

    # 1) Try HDBSCAN first (if installed)
    if HDBSCAN_AVAILABLE and X.shape[0] >= 5:
        try:
            import hdbscan
            min_cluster_sz = max(3, int(0.03 * X.shape[0]))
            hdb = hdbscan.HDBSCAN(min_cluster_size=min_cluster_sz, min_samples=2)
            tmp_labels = hdb.fit_predict(X)
            n_clusters_hdb = len(set(tmp_labels)) - (1 if -1 in tmp_labels else 0)
            if n_clusters_hdb >= 1:
                cluster_labels = tmp_labels
                cluster_probs  = hdb.probabilities_
                cluster_algo   = "HDBSCAN"
                print(f"[{S['label']}] HDBSCAN min_cluster_size={min_cluster_sz} "
                      f"min_samples=2 -> {n_clusters_hdb} cluster(s)")
            else:
                print(f"[{S['label']}] HDBSCAN returned no clusters - "
                      f"falling back to DBSCAN")
        except Exception as e:
            print(f"[{S['label']}] HDBSCAN failed ({e}) - falling back to DBSCAN")

    # 2) Fallback (or primary if HDBSCAN unavailable): DBSCAN
    if cluster_labels is None:
        k = 2
        nbrs = NearestNeighbors(n_neighbors=k).fit(X)
        distances, _ = nbrs.kneighbors(X)
        k_dist = np.sort(distances[:, k-1])[::-1]
        if len(k_dist) >= 4:
            d1 = np.diff(k_dist)
            d2 = np.diff(d1)
            elbow_idx = int(np.argmin(d2)) + 1
            eps_auto  = float(round(k_dist[elbow_idx], 2))
        else:
            eps_auto = 1.3

        dbscan = DBSCAN(eps=eps_auto, min_samples=2)
        cluster_labels = dbscan.fit_predict(X)
        if cluster_algo is None:
            cluster_algo = ("DBSCAN" if HDBSCAN_AVAILABLE
                            else "DBSCAN (hdbscan not installed)")
        else:
            cluster_algo = "DBSCAN (fallback from HDBSCAN)"
        print(f"[{S['label']}] DBSCAN eps={eps_auto:.2f} (min_samples=2)")

    ip_agg["cluster"] = cluster_labels
    if cluster_probs is not None:
        ip_agg["cluster_prob"] = cluster_probs

    # --- Cluster-quality diagnostics (unchanged math, broader applicability)
    _labels   = ip_agg["cluster"].values
    _nonnoise = _labels != -1
    _n_clusters = int(len(set(_labels[_nonnoise])))
    _n_noise    = int((_labels == -1).sum())
    try:
        if _nonnoise.sum() >= 2 and _n_clusters >= 2:
            _sil = float(silhouette_score(X[_nonnoise], _labels[_nonnoise]))
        else:
            _sil = None
    except Exception:
        _sil = None

    # --- Persist to session dict for the dashboard --------------------------
    S["ip_agg"] = ip_agg
    S["_X"] = X
    S["_chosen_contamination"] = best_cont
    S["_hopkins"]          = H
    S["_hopkins_label"]    = H_label
    S["_cluster_algo"]     = cluster_algo
    S["_eps_auto"]         = eps_auto
    S["_min_samples"]      = 2
    S["_min_cluster_size"] = min_cluster_sz
    S["_silhouette"]       = _sil
    S["_n_clusters"]       = _n_clusters
    S["_n_noise"]          = _n_noise

    print(f"[{S['label']}] {cluster_algo}: clusters={_n_clusters} noise={_n_noise} "
          f"silhouette={('n/a' if _sil is None else round(_sil,3))}")
    print(f"[{S['label']}] Anomalies: {ip_agg['anomaly'].sum()} / {len(ip_agg)} | "
          f"Clusters: {ip_agg['cluster'].nunique()}")


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

## 8. שינוי בקוד #3 — Cell 47 (Model Diagnostics card)

Cell 47 הוא תא גדול של styling ו-layout. רק קטע אחד בתוכו משתנה — הפונקציה שמרנדרת
את כרטיס ה-Model Diagnostics בדשבורד.

**מה משתנה:**

- נוספת שורה חדשה בכרטיס: `Hopkins statistic` עם ערך ופרשנות.
- שורת ה-`DBSCAN` משתנה ל-`Clustering ({_algo})` ומציגה את האלגוריתם שהורץ בפועל.
- שורת הפרמטרים דינמית: `eps` כש-DBSCAN, `min_cluster_size` כש-HDBSCAN.

**מה לא משתנה:** כל שאר Cell 47 (palette, NAV_ITEMS, AURORA_INDEX_STRING וכו') —
מוצג כאן רק הקטע הרלוונטי.

### BEFORE — הקטע הנוכחי של Model Diagnostics

</div>


In [ ]:
def _row(model, body):
            return html.Div([
                html.Div(model, style={"fontFamily":"'JetBrains Mono', monospace",
                    "fontSize":"11px","letterSpacing":"0.12em","textTransform":"uppercase",
                    "color":accent,"fontWeight":"700","marginBottom":"3px"}),
                html.Div(body, style={"fontFamily":"'JetBrains Mono', monospace",
                    "fontSize":"12.5px","color":INK_DIM,"lineHeight":"1.65"}),
            ], style={"marginBottom":"12px"})

        cards.append(dbc.Col(html.Div([
            html.Div(f"{lbl} - model diagnostics", style={
                "fontFamily":"'Newsreader', Georgia, serif","fontSize":"1.15rem",
                "color":INK,"fontWeight":"500","marginBottom":"14px"}),
            _row("IsolationForest",
                 f"contamination = {_fmt(s.get('_chosen_contamination'))}  -  "
                 f"anomalies = {n_anom} / {n_ips}"),
            _row("DBSCAN",
                 f"eps = {_fmt(s.get('_eps_auto'))} (k-distance elbow)  -  "
                 f"min_samples = {s.get('_min_samples', 2)}  -  "
                 f"clusters = {s.get('_n_clusters', 'n/a')}  -  "
                 f"noise = {s.get('_n_noise', 'n/a')}  -  "
                 f"silhouette = {_fmt(s.get('_silhouette'), 3)}"),
            _row("LSTM",
                 (f"threshold = {_fmt(thr, 5)} (val_mean + 2*val_std)  -  "
                  f"flagged = {n_lstm} / {n_seq}") if thr is not None
                 else "not trained for this session"),
        ], style={**CARD, "borderRadius":"14px","padding":"18px 20px","height":"100%",
                  "borderLeft":f"3px solid {accent}"}),
            md=6, style={"marginBottom":"14px"}))

    if not cards:
        return html.Div()
    return html.Div([
        html.Div("Model Diagnostics  -  dynamic, recomputed per capture", style={
            "fontFamily":"'JetBrains Mono', monospace","fontSize":"11px",
            "letterSpacing":"0.2em","textTransform":"uppercase","color":VIOLET_BRIGHT,
            "fontWeight":"700","margin":"28px 0 12px"}),
        dbc.Row(cards),
    ])


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

### AFTER — הקטע המוצע

</div>


In [ ]:
def _row(model, body):
            return html.Div([
                html.Div(model, style={"fontFamily":"'JetBrains Mono', monospace",
                    "fontSize":"11px","letterSpacing":"0.12em","textTransform":"uppercase",
                    "color":accent,"fontWeight":"700","marginBottom":"3px"}),
                html.Div(body, style={"fontFamily":"'JetBrains Mono', monospace",
                    "fontSize":"12.5px","color":INK_DIM,"lineHeight":"1.65"}),
            ], style={"marginBottom":"12px"})

        # Build the clustering row dynamically based on which algorithm was used
        _algo = s.get("_cluster_algo") or "DBSCAN"
        if "HDBSCAN" in _algo and "fallback" not in _algo.lower():
            _params = (f"min_cluster_size = {s.get('_min_cluster_size', 'n/a')}  -  "
                       f"min_samples = {s.get('_min_samples', 2)}")
        else:
            _params = (f"eps = {_fmt(s.get('_eps_auto'))} (k-distance elbow)  -  "
                       f"min_samples = {s.get('_min_samples', 2)}")

        cards.append(dbc.Col(html.Div([
            html.Div(f"{lbl} - model diagnostics", style={
                "fontFamily":"'Newsreader', Georgia, serif","fontSize":"1.15rem",
                "color":INK,"fontWeight":"500","marginBottom":"14px"}),
            _row("IsolationForest",
                 f"contamination = {_fmt(s.get('_chosen_contamination'))}  -  "
                 f"anomalies = {n_anom} / {n_ips}"),
            _row("Hopkins statistic",
                 f"H = {_fmt(s.get('_hopkins'), 3)}  ->  "
                 f"{s.get('_hopkins_label', 'n/a')}"),
            _row(f"Clustering ({_algo})",
                 f"{_params}  -  "
                 f"clusters = {s.get('_n_clusters', 'n/a')}  -  "
                 f"noise = {s.get('_n_noise', 'n/a')}  -  "
                 f"silhouette = {_fmt(s.get('_silhouette'), 3)}"),
            _row("LSTM",
                 (f"threshold = {_fmt(thr, 5)} (val_mean + 2*val_std)  -  "
                  f"flagged = {n_lstm} / {n_seq}") if thr is not None
                 else "not trained for this session"),
        ], style={**CARD, "borderRadius":"14px","padding":"18px 20px","height":"100%",
                  "borderLeft":f"3px solid {accent}"}),
            md=6, style={"marginBottom":"14px"}))

    if not cards:
        return html.Div()
    return html.Div([
        html.Div("Model Diagnostics  -  dynamic, recomputed per capture", style={
            "fontFamily":"'JetBrains Mono', monospace","fontSize":"11px",
            "letterSpacing":"0.2em","textTransform":"uppercase","color":VIOLET_BRIGHT,
            "fontWeight":"700","margin":"28px 0 12px"}),
        dbc.Row(cards),
    ])


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

## 9. ערובות תאימות לאחור

<table dir="rtl" style="border-collapse:collapse; width:100%;">
<thead>
<tr style="background:#eef;">
<th style="border:1px solid #ccc; padding:6px;">סיכון</th>
<th style="border:1px solid #ccc; padding:6px;">מיטיגציה במימוש</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #ccc; padding:6px;"><code>hdbscan</code> לא ניתן להתקנה במחשב המשתמש</td>
<td style="border:1px solid #ccc; padding:6px;">דגל <code>HDBSCAN_AVAILABLE=False</code> + fallback אוטומטי. אין הפסקת ריצה.</td>
</tr>
<tr>
<td style="border:1px solid #ccc; padding:6px;">HDBSCAN רץ אך מחזיר כל-noise</td>
<td style="border:1px solid #ccc; padding:6px;">תנאי <code>n_clusters_hdb &gt;= 1</code> מזהה זאת ועובר ל-DBSCAN.</td>
</tr>
<tr>
<td style="border:1px solid #ccc; padding:6px;">סכמת <code>ip_agg["cluster"]</code> משתנה</td>
<td style="border:1px solid #ccc; padding:6px;">לא משתנה. שני האלגוריתמים מחזירים <code>{−1, 0, 1, …}</code>.</td>
</tr>
<tr>
<td style="border:1px solid #ccc; padding:6px;">Model Agreement Matrix נשבר</td>
<td style="border:1px solid #ccc; padding:6px;">לא — הוא צופה רק את <code>cluster == -1</code> / <code>!= -1</code>.</td>
</tr>
<tr>
<td style="border:1px solid #ccc; padding:6px;">חישוב Silhouette מפסיק לעבוד</td>
<td style="border:1px solid #ccc; padding:6px;">אותו חישוב — דורש ≥ 2 clusters non-noise. ב-HDBSCAN יותר סביר שיהיו.</td>
</tr>
</tbody>
</table>

</div>


<div dir="rtl" style="text-align:right; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:15px;">

## 10. שש שאלות מוכנות למרצה

### שאלה #1 — האם DBSCAN הוא בכלל המודל הנכון?

ה-DBSCAN שלי בוחר `eps` אוטומטית מ-k-distance elbow ויוצא 0.78 בסשן אחד ו-4.86 בסשן אחר
על אותה רשת. שניהם מקבצים את כל ה-IP-ים ל-cluster יחיד + מעט noise. ה-Silhouette לא מוגדר
במצב כזה. בנתונים האלה — האם DBSCAN בכלל המודל הנכון, או שהפיצ'רים שלי לא מפרידים מספיק
ואני צריך GMM / HDBSCAN / Mean-Shift?

**תשובה צפויה:** התופעה תקפה ברשת רגילה. אם רוצים אשכולות התנהגותיים מובחנים — HDBSCAN
עדיף (לא דורש eps, מטפל בצפיפויות משתנות). GMM ייכשל כי תעבורת רשת skewed. מומלץ קודם
לחשב Hopkins.

### שאלה #2 — `eps` דינמי לכל סשן

ההיגיון של הקוד שלי: לכל סשן נפרד מחשב k-distance על k=2, מוצא את המרפק. האם זה לגיטימי
לכייל `eps` נפרד לכל סשן? מצד אחד זה מתאים לצפיפות הנתונים — מצד שני זה אומר שאני לא יכול
להשוות חד-משמעית בין S1 ל-S2.

**תשובה צפויה:** כן, לגיטימי עם תיעוד. שלוש אסכולות: (א) eps נפרד adaptive, (ב) eps אחיד
לכל הסשנים להשוואה, (ג) היברידי. מאמרי הקהילה משווים רק מאפיינים אגרגטיביים (כמה noise,
כמה clusters) ולא תוויות.

### שאלה #3 — ערך ה-fallback של 1.3

אם יש פחות מ-4 IP-ים — fallback ל-`eps=1.3`. מאיפה הערך הזה?

**תשובה צפויה:** ערך אמפירי שמתאים למרחב נירמל ע"י StandardScaler. רלוונטי רק לקצוות
(פחות מ-4 IP-ים) שבהם ממילא אין מה לאשכל. Ester et al. (1996) ממליצים על percentile 90.

### שאלה #4 — Silhouette כשיש cluster יחיד

ב-Model Diagnostics רואים `Silhouette = n/a` כי Silhouette לא מוגדר ל-cluster יחיד.
אני עדיין רוצה למדוד איכות ההפרדה בין cluster ל-noise. מה השמדן הנכון?

**תשובה צפויה:** **DBCV** (Density-Based Clustering Validation) — המדד הטוב ביותר
לקלאסטרינג מבוסס-צפיפות, כי הוא לוקח noise בחשבון. Davies-Bouldin / Calinski-Harabasz
דורשים 2+ clusters ולא יעזרו. cluster יחיד + 6 noise הוא ממצא משמעותי אם Hopkins > 0.7.

### שאלה #5 — שילוב 3 מודלים

מה ההצדקה התיאורטית לשילוב IsolationForest + DBSCAN + LSTM? לפי המחקרים העדכניים —
autoencoder יחיד היה תופס את כולם?

**תשובה צפויה:** "No Free Lunch in Anomaly Detection" (Aggarwal, 2017): כל מודל מניח
הנחה שונה. IF = point, DBSCAN = contextual, LSTM = temporal/collective. החפיפה
מינימלית. autoencoder תופס שניים אבל קופסה שחורה — באבטחה explainability קריטי.

### שאלה #6 — הערכה אקדמית בלי תוויות

בלי תוויות אני לא יכול למדוד precision/recall. ההערכה שלי מסתמכת על Hopkins + Silhouette
+ הסכמה בין המודלים. האם הזרקת התקפות סינתטיות ידועות לתוך ה-PCAP מקובלת אקדמית?

**תשובה צפויה:** כן — נקרא **semi-synthetic ground truth injection**. CIC-IDS2017
ו-UNSW-NB15 נוצרו כך. כלים: Scapy, nmap, dnscat2. ציטוטים: Sharafaldin et al. (2018),
Ring et al. (2019).

</div>
